In [4]:
!unzip -o application_train.csv.zip

import pandas as pd
df = pd.read_csv("application_train.csv")
print(df.shape)

Archive:  application_train.csv.zip
  inflating: application_train.csv   
(307511, 122)


In [5]:
import numpy as np

df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
print("Persentase baris penanda:", df["DAYS_EMPLOYED_ANOM"].mean())
print(df.groupby("DAYS_EMPLOYED_ANOM")["TARGET"].mean())

df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365

Persentase baris penanda: 0.18007160719453938
DAYS_EMPLOYED_ANOM
0    0.086600
1    0.053996
Name: TARGET, dtype: float64


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

X = df.drop(columns=["TARGET", "SK_ID_CURR"])
y = df["TARGET"]

for c in X.select_dtypes("object").columns:
    X[c] = X[c].astype("category")

X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05,
                           random_state=42, verbose=-1)
model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="auc",
          callbacks=[lgb.early_stopping(50)])

print("AUC validasi:", roc_auc_score(y_va, model.predict_proba(X_va)[:, 1]))

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[247]	valid_0's auc: 0.760158	valid_0's binary_logloss: 0.245241
AUC validasi: 0.7601575420030489
